<html><div style="font-size:7pt">This notebook may contain text, code and images generated by artificial intelligence. Used model: claude-sonnet-4-20250514, vision model: claude-sonnet-4-20250514, endpoint: None, bia-bob version: 0.34.3.. It is good scientific practice to check the code and results it produces carefully. <a href="https://github.com/haesleinhuepf/bia-bob">Read more about code generation using bia-bob</a></div></html>

# Image Embeddings and UMAP Generation

This notebook demonstrates how to:
1. Generate vision embeddings using the [openai/clip-vit-base-patch32](https://huggingface.co/openai/clip-vit-base-patch32)
2. Create a 3D UMAP visualization of the embeddings
3. Save the results for VEST

In [1]:
import os
import numpy as np
import pandas as pd
from PIL import Image
import torch
import torch.nn.functional as F
from transformers import AutoImageProcessor, AutoModel
from umap import UMAP
import random
from pathlib import Path
import requests
from io import BytesIO
import torch

In [2]:
import transformers
transformers.__version__

'5.12.1'

In [3]:
base_dir = Path("./")
images_dir = base_dir / "images"

## Load Vision Embedding Model

In [4]:
from transformers import CLIPProcessor, CLIPModel
import pandas as pd
import torch
from PIL import Image
import requests
import os

# Initialize models
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

config.json:   0%|          | 0.00/4.19k [00:00<?, ?B/s]

C:\Users\haase\miniforge3\envs\bob-env\Lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\haase\.cache\huggingface\hub\models--openai--clip-vit-base-patch32. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/862k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/525k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.22M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

## Generate Vision Embeddings for All Images

In [5]:
import os
import numpy as np
import pandas as pd
from PIL import Image
import torch
import torch.nn.functional as F

# Get list of all image files in the images folder and subfolders
image_files = []
for root, dirs, files in os.walk(images_dir):
    for f in files:
        if f.lower().endswith('.png') or f.lower().endswith('.jpg'):
            image_files.append(os.path.join(root, f))



print(f"{len(image_files)} images found")

210 images found


In [6]:
#image_files = random.sample(image_files, 500)

In [7]:
print(f"Processing {len(image_files)} images for embeddings...")

# Initialize lists to store results
filenames = []
embeddings = []
images = []

# Loop through all image files
for i, image_path in enumerate(image_files):
    # Obtain filename from path
    filename = os.path.relpath(image_path, images_dir)

    # Load the image
    current_image = Image.open(image_path)
    images.append(np.asarray(current_image))

    # Apply the processing pipeline
    current_inputs = clip_processor(images=current_image, return_tensors="pt")
    with torch.no_grad():
        current_np_emb = clip_model.get_image_features(**current_inputs)[0].cpu().squeeze()[0].tolist()

    # Store results
    filenames.append(filename)
    embeddings.append(current_np_emb)

    if (i + 1) % 100 == 0:
        print(f"Processed {i + 1}/{len(image_files)} images")

# Create DataFrame
df = pd.DataFrame({
    'filename': filenames,
    'embedding': embeddings
})

# Add a new column 'layout' based on the filename
df['layout'] = df['filename'].apply(lambda x: 1 if '_UL_' in x else 2)

print(f"Successfully processed {len(df)} images")
display(df.head())

Processing 210 images for embeddings...
Processed 100/210 images
Processed 200/210 images
Successfully processed 210 images


,filename,embedding,layout
0,11_RDM_page01.png,"[-0.05880768597126007, 0.001273319125175476, 0...",2
1,11_RDM_page02.png,"[0.010850191116333008, 0.17468787729740143, 0....",2
2,11_RDM_page03.png,"[-0.0703616738319397, 0.17920269072055817, 0.0...",2
3,11_RDM_page04.png,"[0.03190414607524872, 0.25511622428894043, 0.0...",2
4,11_RDM_page05.png,"[-0.02796362340450287, 0.2169242799282074, -0....",2


## Create 3D UMAP Visualization from Embeddings

In [8]:
# Convert embeddings list to numpy array matrix
embedding_matrix = np.stack(df['embedding'].values)

print(f"Embedding matrix shape: {embedding_matrix.shape}")
print("Creating 3D UMAP coordinates...")

# Create 3D UMAP
umap_reducer = UMAP(n_components=3, random_state=42)
umap_coords_actual = umap_reducer.fit_transform(embedding_matrix)

# Add UMAP coordinates to dataframe
df['x'] = umap_coords_actual[:, 0]
df['y'] = umap_coords_actual[:, 1]
df['z'] = umap_coords_actual[:, 2]

print("UMAP coordinates created successfully")
print(f"X range: {df['x'].min():.2f} to {df['x'].max():.2f}")
print(f"Y range: {df['y'].min():.2f} to {df['y'].max():.2f}")
print(f"Z range: {df['z'].min():.2f} to {df['z'].max():.2f}")
display(df.head())

Embedding matrix shape: (210, 768)
Creating 3D UMAP coordinates...


C:\Users\haase\miniforge3\envs\bob-env\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


UMAP coordinates created successfully
X range: -1.19 to 6.35
Y range: -1.29 to 3.97
Z range: 3.79 to 8.02


,filename,embedding,layout,x,y,z
0,11_RDM_page01.png,"[-0.05880768597126007, 0.001273319125175476, 0...",2,0.908017,0.603683,6.068271
1,11_RDM_page02.png,"[0.010850191116333008, 0.17468787729740143, 0....",2,0.748177,0.632403,6.475627
2,11_RDM_page03.png,"[-0.0703616738319397, 0.17920269072055817, 0.0...",2,0.237475,-1.157015,6.497894
3,11_RDM_page04.png,"[0.03190414607524872, 0.25511622428894043, 0.0...",2,0.144671,-1.233541,6.464936
4,11_RDM_page05.png,"[-0.02796362340450287, 0.2169242799282074, -0....",2,0.209207,-1.191703,6.571118


## Save Results to CSV File

In [9]:
# Save the dataframe to CSV
output_path = base_dir / "data.csv"

# Convert embedding arrays to strings for CSV storage
df_to_save = df.copy()
df_to_save['embedding'] = df_to_save['embedding'].apply(lambda x: ','.join(map(str, x)))

df_to_save.to_csv(output_path, index=False)

print(f"Results saved to {output_path}")
print(f"Final dataframe shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
display(df[['filename', 'x', 'y', 'z']].head(10))

Results saved to data.csv
Final dataframe shape: (210, 6)
Columns: ['filename', 'embedding', 'layout', 'x', 'y', 'z']


,filename,x,y,z
0,11_RDM_page01.png,0.908017,0.603683,6.068271
1,11_RDM_page02.png,0.748177,0.632403,6.475627
2,11_RDM_page03.png,0.237475,-1.157015,6.497894
3,11_RDM_page04.png,0.144671,-1.233541,6.464936
4,11_RDM_page05.png,0.209207,-1.191703,6.571118
5,11_RDM_page06.png,0.932507,0.874533,6.541903
6,11_RDM_page07.png,0.809902,1.050264,6.628998
7,11_RDM_page08.png,0.264686,-1.225449,6.611201
8,11_RDM_page09.png,0.053080,-1.292512,6.569473
9,11_RDM_page10.png,0.085861,0.665724,6.240201
